In [88]:
#create subarea shapefile with subbasin and field

In [89]:
import geopandas as gpd
import pandas as pd
import os
from rasterstats import zonal_stats
import numpy as np
import whitebox
from geocube.api.core import make_geocube

In [90]:
#file path

#existing, wetland loss and wetland restoration
field_shp = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_NoBuffer/field_wetland_no_buffer.shp"
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_NoBuffer/output"

#wetland enhancement 5m
field_shp = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_5m/field_wetland_enhancement_5m.shp"
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_5m/output"

#wetland enhancement 15m
field_shp = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_15m/field_wetland_enhancement_15m.shp"
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_15m/output"

#wetland enhancement 30m
field_shp = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_30m/field_wetland_enhancement_30m.shp"
output_path = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_30m/output"


#don't change from here for same scenario
#----------------------------------------------------------------------------------------
subbasin_shp = "D:/imWEBs/BRC/model/BRC/Watershed/Output/subbasin.shp"
dem_tif = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/dem.tif"
slope_tif = "D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea/slope.tif"

#used to remove subareas that are too small which depends on cell size
subarea_min_area_ha = 0.0001

#don't change anything from here
#----------------------------------------------------------------------------------------
#id column
subbasin_id_col = "VALUE"
field_id_col = "LAND_ID"        #the field id in field shapefile

#output files
subarea_shp = os.path.join(output_path, 'subarea.shp')
subarea_csv = os.path.join(output_path, 'subarea.csv')
subarea_tif = os.path.join(output_path, "subarea.tif")
subarea_dep = os.path.join(output_path, "subarea.dep")

#fixed column name, don't change
subarea_id_col = "Id"
subarea_field_id_col = "FieldId"
subarea_subbasin_id_col = "SubbasinId"
subarea_area_col = "Area"
subarea_slope_col = "Slope"
subarea_elevation_col = "Elevation"

In [91]:
def generate_subaera():
    """
    generate subarea shapefile with subbasin and field shapefile

    Returns
    -------
    DataFrame for subarea
    
    """
    #read shapefile and restructure the columns
    field_shp_df = gpd.read_file(field_shp)
    field_shp_df[subarea_field_id_col] = field_shp_df[field_id_col].astype(int)
    field_shp_df.sort_values(by = subarea_field_id_col, inplace=True)

    subbasin_shp_df = gpd.read_file(subbasin_shp)
    subbasin_shp_df[subarea_subbasin_id_col] = subbasin_shp_df[subbasin_id_col].astype(int)
    subbasin_shp_df.sort_values(by = subarea_subbasin_id_col, inplace=True)

    #identify
    subarea_shp_df = gpd.overlay(subbasin_shp_df, field_shp_df, how='identity')

    #add area in ha
    subarea_shp_df[subarea_area_col] = subarea_shp_df.area / 10000

    #remove subareas that are two small
    subarea_shp_df = subarea_shp_df[subarea_shp_df[subarea_area_col] > subarea_min_area_ha]

    #add subarea id
    subarea_shp_df.reset_index(inplace=True)
    subarea_shp_df[subarea_id_col] = subarea_shp_df.index + 1

    #save to shapefile
    subarea_shp_df.to_file(subarea_shp)

    #return the dataframe
    return subarea_shp_df

In [92]:
def get_subarea_raster_average(subarea_shp, raster, col_name):
    """
    get average value for each subbarea

    Parameters
    ----------
    subarea_shp : the file path of subarea shapefile
    raster      : the file path of statistic raster 
    col_name    : the column name

    Returns
    -------
    DataFrame with two columns:
    1. SubAreaId - SubArea ID
    2. col_name - The statistic value
    
    """
    stats = zonal_stats(subarea_shp, raster, band=1,stats='mean', geojson_out = True)

    subarea_mean = {}
    for land in stats:
        #skip if majority is none
        if land['properties'][subarea_id_col] is None:
            continue
        elif land['properties']['mean'] is None:
            subarea_mean[int(land['properties'][subarea_id_col])] = None
        else:
            subarea_mean[int(land['properties'][subarea_id_col])] = land['properties']['mean']
    
    #create dataframe and save the original crop id to Original ID column
    df = pd.DataFrame.from_dict(subarea_mean, orient = 'index', columns = [col_name])
    df.index.name = subarea_id_col
    df = df.sort_index()
    df = df.reset_index()

    #fill the null values
    df = df.fillna(method="ffill")

    return df

In [93]:
#step 1: create the subarea shapefile
print("generating subbarea")
subarea_shp_df = generate_subaera()

#step 2: calculate mean elevation and slope
print("calculate mean elevation and slope")
elevation_df = get_subarea_raster_average(subarea_shp, dem_tif, subarea_elevation_col)
subarea_shp_df = subarea_shp_df.merge(elevation_df, on = subarea_id_col, how = "left")

slope_df = get_subarea_raster_average(subarea_shp, slope_tif, subarea_slope_col)
subarea_shp_df = subarea_shp_df.merge(slope_df, on = subarea_id_col, how = "left")

#save to subarea.csv
subarea_shp_df.to_csv(subarea_csv, columns=[subarea_id_col, subarea_subbasin_id_col, subarea_field_id_col, subarea_area_col, subarea_slope_col, subarea_elevation_col],index = False)


generating subbarea
calculate mean elevation and slope


In [94]:
#create subarea tif from shapefile
out_grid = make_geocube(
    vector_data= subarea_shp,
    measurements=[subarea_id_col],
    resolution=(-30, 30),
)
out_grid[subarea_id_col].rio.to_raster(subarea_tif)

In [95]:
#create subarea.dep
wbt = whitebox.WhiteboxTools()
wbt.resample(subarea_tif,subarea_dep, method='nn', base=dem_tif)

.\whitebox_tools.exe --run="Resample" --inputs='D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_30m/output\subarea.tif' --output='D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/SubArea_Enhancement_30m/output\subarea.dep' --base='D:/git/imWEBs/IMWEBsInterface/resources/DemoDatasets/BRC2021/dem.tif' --method=nn

****************************
* Welcome to Resample      *
* Powered by WhiteboxTools *
* www.whiteboxgeo.com      *
****************************
Reading data...
Progress: 0%
Progress: 1%
Progress: 2%
Progress: 3%
Progress: 4%
Progress: 5%
Progress: 6%
Progress: 7%
Progress: 8%
Progress: 9%
Progress: 10%
Progress: 11%
Progress: 12%
Progress: 13%
Progress: 14%
Progress: 15%
Progress: 16%
Progress: 17%
Progress: 18%
Progress: 19%
Progress: 20%
Progress: 21%
Progress: 22%
Progress: 23%
Progress: 24%
Progress: 25%
Progress: 26%
Progress: 27%
Progress: 28%
Progress: 29%
Progress: 30%
Progress: 31%
Progress: 32%
Progress: 33%
Progress: 34%


0

In [96]:
#create subarea.dep
#wbt = whitebox.WhiteboxTools()

#subarea_soil_intersect_shp = os.path.join(output_path, "subarea_soil_intersect.shp")
#subarea_soil_symmetrical_difference_shp = os.path.join(output_path, "subarea_soil_symmetrical_difference.shp")
#subarea_soil_shp = os.path.join(output_path, "subarea_soil.shp")
#wbt.intersect(subarea_shp, soil_shp, subarea_soil_intersect_shp)
#wbt.symmetrical_difference(subarea_shp, soil_shp, subarea_soil_symmetrical_difference_shp)
#wbt.union(subarea_soil_intersect_shp, subarea_soil_symmetrical_difference_shp, subarea_soil_shp)